In [ ]:
import pandas as pd
import json

def convert_parquet_to_geojson(parquet_file_path, output_file_path):
    print(f"Loading {parquet_file_path}...")
    # 1. Read the Parquet file directly instead of CSV
    df = pd.read_parquet(parquet_file_path)

    # 2. Drop rows missing coordinates right away
    df = df.dropna(subset=['latitude', 'longitude'])

    # 3. Filter for only your requested columns
    desired_columns = [
        'name', 'latitude', 'longitude', 'address', 'locality', 
        'region', 'postcode', 'date_created', 'date_closed', 
        'tel', 'website', 'email', 'fsq_category_ids', 'fsq_category_labels'
    ]
    # Ensure we only keep columns that actually exist in the file
    existing_columns = [col for col in desired_columns if col in df.columns]
    df = df[existing_columns]

    # Clean up "null" strings often found in Foursquare exports
    df = df.replace('null', None)

    print(f"Processing {len(df)} rows into GeoJSON format...")
    features = []

    for _, row in df.iterrows():
        # GeoJSON requires [longitude, latitude] order
        try:
            lon = float(row['longitude'])
            lat = float(row['latitude'])
        except (TypeError, ValueError):
            # Skip records without valid coordinates
            continue

        # Create the properties dictionary (all columns except coordinates)
        # Using .dropna() here ensures your GeoJSON doesn't get bloated with empty fields
        properties = row.drop(['latitude', 'longitude']).dropna().to_dict()

        # Build the GeoJSON Feature
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]
            },
            "properties": properties
        }
        features.append(feature)

    # Build the FeatureCollection
    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    print("Writing to file (this may take a moment)...")
    # Write to file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, ensure_ascii=False, indent=4)

    print(f"Successfully created {output_file_path} with {len(features)} features.")

# Usage
if __name__ == "__main__":
    convert_parquet_to_geojson('au_locations.parquet', 'Data.geojson')